# Kwantowy model Isinga

### wprowadzenie
Do matematycznego opisu kwantowego modelu potrzebujemy *macierzy Pauliego*. Są to macierze $2 \times 2$ które opisują interakcje spinu z polem magnetycznym. Konkrektnie, będziemy potrzebować macierzy $\sigma^z$ oraz $\sigma^x$ zdefiniowanych jako:

$$
\sigma^z = \begin{bmatrix} 1 & 0 \\ 0 & -1 \end{bmatrix}, \quad \sigma^x = \begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix} 
$$ 

Ponadto będziemy potrzebować wektorów i własności własnych. Dla przypomnienia, jeżeli $A$ jest macierzą, to wektor $v$ nazywamy **wektorem własnym** jeżeli spełniony jest warunek:

$$
Av = \lambda v
$$

gdzie $\lambda$ jest niezerowym skalarem nazywanym **wartością własną**.

## Definicja
Kwantowy model Isinga jest definiowany w następujący sposób:

$$
\mathcal{H} =  \sum_{(i, j)} J_{ij} \sigma_i^z \sigma_j^z + \sum_{i=1}^{N} h_i \sigma_i^z
$$


Warto pamiętać, że w przypadku działania macierzy Pauliego $\sigma^{\alpha}$ ($\alpha = x,z$) na spin $i$, zapis $\sigma^{\alpha}_i$ oznacza

\begin{equation*} \sigma^{\alpha}_i = I \otimes I \otimes \ldots \otimes I  \otimes \sigma^{\alpha} \otimes I \otimes \ldots \otimes I \end{equation*}

gdzie $\otimes$ jest iloczynem tensorowym (lub iloczynem Kroneckera), z $i$-tym czynnikiem  $\sigma^{\alpha}$.

Iloczyn tensorowy jest definiowany następująco:
$$
A \otimes B \;=\;
\begin{bmatrix}
a_{11}B & a_{12}B & \cdots & a_{1n}B\\
a_{21}B & a_{22}B & \cdots & a_{2n}B\\
\vdots  & \vdots  & \ddots & \vdots \\
a_{m1}B & a_{m2}B & \cdots & a_{mn}B
\end{bmatrix}.

$$
Przykład dla macierzy 2x2

$$
A=\begin{bmatrix}
a_{11} & a_{12}\\
a_{21} & a_{22}
\end{bmatrix},\quad
B=\begin{bmatrix}
b_{11} & b_{12}\\
b_{21} & b_{22}
\end{bmatrix},
\quad
A\otimes B=
\begin{bmatrix}
a_{11}B & a_{12}B\\
a_{21}B & a_{22}B
\end{bmatrix}
=
\begin{bmatrix}
a_{11}b_{11} & a_{11}b_{12} & a_{12}b_{11} & a_{12}b_{12}\\
a_{11}b_{21} & a_{11}b_{22} & a_{12}b_{21} & a_{12}b_{22}\\
a_{21}b_{11} & a_{21}b_{12} & a_{22}b_{11} & a_{22}b_{12}\\
a_{21}b_{21} & a_{21}b_{22} & a_{22}b_{21} & a_{22}b_{22}
\end{bmatrix}
$$

### Związek między klasycznym a kwantowym modelem Isinga

Hamiltonian $\mathcal{H}$ jest olbrzymią macierzą o rozmiarze $2^n \times 2^n$, gdzie $n$ jest ilością spinów, natomiast klasyczny $H$, to jedynie pewna suma wskazująca wartość energii dla danego stanu. Jak, tak różne obiekty (liczba i macierz) mają się do siebie? Odpowiedź leży w specyfice mechaniki kwantowej.

W pewnym uproszczeniu, jeżeli Hamiltonian opisujący układ kwantowy jest macierzą, to wektory własne tej macierzy opisują stany, które ten układ może przyjąć a odpowiadające im własności własne są energiami tych stanów.

Przyjmijmy notację:
$$
\ket{\uparrow} = \begin{bmatrix} 1 \\ 0 \end{bmatrix} \quad \ket{\downarrow} = \begin{bmatrix} 0 \\ 1 \end{bmatrix} 
$$

Można pokazać, że wektory własne macierzy $\sigma^z$ to właśnie $\ket{\uparrow}$ i $\ket{\downarrow}$ z odpowiadającymi im wartościami własnymi $+1$ i $-1$. Zatem macierz $\sigma^z$ opisuje pojedyńczy spin. 

W ten sam sposób, wektory i własności własne kwantowego modelu Isinga bez pola poprzecznego odpowiadają stanom i energią w klasycznym modelu Isinga.


In [1]:
# Oba modele Isinga - klasyczny i kwantowy dają te same stany
# pokażemy to wprost na bardzo małym przykładzie (na razie ignorując pole poprzeczne)

import numpy as np
import qutip as qt
from itertools import product

J = {(1, 2): 0.5, (1, 3): -1, (2, 3): 0.75}
h = {1: 1, 2: -1, 3: 0.5}

# kwantowy
sigma_1 = qt.tensor([qt.sigmaz(), qt.qeye(2), qt.qeye(2)])
sigma_2 = qt.tensor([qt.qeye(2), qt.sigmaz(), qt.qeye(2)])
sigma_3 = qt.tensor([qt.qeye(2), qt.qeye(2), qt.sigmaz()])

H_quantum = J[(1, 2)] * sigma_1 * sigma_2 + J[(1, 3)] * sigma_1 * sigma_3 + J[(2, 3)] * sigma_2 * sigma_3 + \
            h[1] * sigma_1 + h[2] * sigma_2 + h[3] * sigma_3

eigenvalues, eigenstates = H_quantum.eigenstates()

def decode_eigenstate(eigenstate):
    state = eigenstate.full()
    idx, _ = np.nonzero(state)
    idx = idx.item()
    basis_states = list(product([1,-1], repeat=3))
    return basis_states[idx]

# klasyczny
energies_states = {}

for state in product([-1,1], repeat=3):
    energy = sum([state[i-1] * state[j-1] * J[(i, j)] for  (i, j) in J.keys()]) + sum([state[i] * h[i+1] for i in range(3)])

    energies_states[energy] = state

energies_states = dict(sorted(energies_states.items()))

# podsumowanie
print("Wartości własne macierzy H:", eigenvalues)
print("Wszystkie energie klasycznego modelu:", list(energies_states.keys()))
print("------------")
print("Stan odpowiadający najmiejszej wartości własnej", decode_eigenstate(eigenstates[0]))
print("Stan podstawowy klasycznego modelu: ", energies_states[-4.75])

Wartości własne macierzy H: [-4.75 -0.25 -0.25  0.25  0.25  0.75  1.25  2.75]
Wszystkie energie klasycznego modelu: [-4.75, -0.25, 0.25, 0.75, 1.25, 2.75]
------------
Stan odpowiadający najmiejszej wartości własnej (-1, 1, -1)
Stan podstawowy klasycznego modelu:  (-1, 1, -1)


### Kwantowy model Isinga z polem poprzecznym 

Na razie kwantowy model Isinga jest tylko wyłącznie innym matematycznym opisem tego samego obiektu.  

Kwantowy model Isinga z polem poprzecznym jest definiowany w następujący sposób:

$$
\mathcal{H} = \Gamma \sum_{i=1}^N \sigma^x_i + \sum_{(i, j)} J_{ij} \sigma_i^z \sigma_j^z + \sum_{i=1}^{N} h_i \sigma_i^z
$$

gdzie $\Gamma$ jest parametrem określającym siłę pola poprzecznego.